# Scrape Happiness / Joy Data from Reddit

Collects posts from 15 positive subreddits using Reddit's public `.json` feed — no API key required.

**Subreddits:** r/happy, r/Happiness, r/joy, r/aww, r/wholesomememes, r/UpliftingNews, r/GetMotivated,
r/CasualConversation, r/Mindfulness, r/gratitude, r/selfimprovement, r/mildlyinteresting, r/Wellbeing, r/funny, r/todayilearned

Output: `data/raw/happiness_reddit.csv` with `label=0` (not distressed), targeting ~15,000 rows.

In [ ]:
import requests
import pandas as pd
import time
from datetime import datetime
from pathlib import Path

In [ ]:
# --- Config ---
SUBREDDITS = [
    # Original positive subreddits
    "happy", "Happiness", "joy", "aww", "wholesomememes",
    # Additional anti-depression / uplifting subreddits
    "UpliftingNews", "GetMotivated", "CasualConversation",
    "Mindfulness", "gratitude", "selfimprovement",
    "mildlyinteresting", "Wellbeing", "funny", "todayilearned",
]
MAX_POSTS_PER_SUBREDDIT = 3000  # increased to target ~15k total
SLEEP_BETWEEN_REQUESTS = 1    # seconds — be polite to Reddit

OUTPUT_PATH = Path("../../data/raw/happiness_reddit.csv")

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
                  "AppleWebKit/537.36 (KHTML, like Gecko) "
                  "Chrome/122.0.0.0 Safari/537.36"
}

In [ ]:
def scrape_subreddit(name: str, max_posts: int = 300) -> list:
    """Scrape posts from a subreddit using Reddit's public .json feed."""
    posts = []
    after = None
    base_url = f"https://www.reddit.com/r/{name}.json"

    print(f"Scraping r/{name} ...", end="")

    while len(posts) < max_posts:
        params = {"limit": 100}
        if after:
            params["after"] = after

        try:
            response = requests.get(base_url, headers=HEADERS, params=params, timeout=15)
        except requests.RequestException as e:
            print(f" request error: {e}")
            break

        if response.status_code == 429:
            print(" rate limited, waiting 30s ...")
            time.sleep(30)
            continue
        if response.status_code != 200:
            print(f" HTTP {response.status_code}, stopping.")
            break

        data = response.json().get("data", {})
        children = data.get("children", [])

        if not children:
            break

        for child in children:
            post = child.get("data", {})
            if post.get("stickied") or post.get("author") == "[deleted]":
                continue
            body = post.get("selftext", "") or ""
            if body in ("[deleted]", "[removed]"):
                body = ""
            posts.append({
                "subreddit": name,
                "title": post.get("title", ""),
                "body": body,
                "score": post.get("score", 0),
                "num_comments": post.get("num_comments", 0),
                "created_utc": datetime.utcfromtimestamp(
                    post.get("created_utc", 0)
                ).strftime("%Y-%m-%d %H:%M:%S"),
            })

        after = data.get("after")
        print(f" {len(posts)}", end="", flush=True)

        if not after:
            break

        time.sleep(SLEEP_BETWEEN_REQUESTS)

    print(f" -> {len(posts)} posts collected")
    return posts

In [ ]:
# --- Scrape all subreddits ---
all_posts = []

for subreddit in SUBREDDITS:
    posts = scrape_subreddit(subreddit, max_posts=MAX_POSTS_PER_SUBREDDIT)
    all_posts.extend(posts)

print(f"\nTotal posts collected: {len(all_posts)}")

In [ ]:
# --- Build DataFrame ---
df = pd.DataFrame(all_posts)

# Merge title + body into a single `text` column (matches ML pipeline format)
df["text"] = (df["title"].fillna("") + " " + df["body"].fillna("")).str.strip()

# Assign label=0 (happy/positive = not depressed)
df["label"] = 0

# Drop rows where text is empty
df = df[df["text"].str.len() > 0].reset_index(drop=True)

# Deduplicate on text
before = len(df)
df = df.drop_duplicates(subset="text").reset_index(drop=True)
print(f"Removed {before - len(df)} duplicates. Final shape: {df.shape}")

df.head()

In [ ]:
# --- Stats ---
print("Posts per subreddit:")
print(df["subreddit"].value_counts())
print("\nLabel distribution:")
print(df["label"].value_counts())
print(f"\nAverage text length: {df['text'].str.len().mean():.0f} chars")

In [ ]:
# --- Save to CSV ---
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(OUTPUT_PATH, index=False)
print(f"Saved {len(df)} rows to {OUTPUT_PATH.resolve()}")